
# Local-GPU Dataset Build — Pilot Model Comparison

- Coder: Lenin G. Falconi

## Goal

Extend the SQuAD-style robbery QA dataset from `dataset_build.ipynb` (1,540/174,594 contexts, built via the Hugging Face Inference API with Meta-Llama-3-8B-Instruct) using **local GPU** inference instead, with a different model (not Llama-3-8B, already evaluated).

This supports an ablation hypothesis: *does fine-tuning extractive QA improve with more training samples?* The existing 1,540-sample dataset (`dataset_fge_squadM1.json`) stays as the **baseline**; this notebook builds a **pilot** to compare candidate local models before scaling up with `scripts/build_dataset_local_gpu.py`.

## This notebook

1. Reuses the same 82–150 word context filter as `dataset_build.ipynb` (chosen to exclude outliers, not an API limitation — kept unchanged)
2. Loads two candidate models simultaneously, 4-bit quantized, one per local GPU (12GB / 15GB VRAM):
   - `Qwen/Qwen2.5-7B-Instruct` on `cuda:0`
   - `google/gemma-3-4b-it` on `cuda:1`
3. Runs both over the same small pilot subset via `scripts/local_qa_generation.py` (shared with the full-scale script), using `outlines` for schema-constrained JSON generation
4. Compares the two models' output quality using the same analysis patterns as `dataset_xplore_and_upload.ipynb`

**Note:** this notebook has not been executed in this environment (no local GPU here). Run and validate it on the GPU-equipped machine, then report results back.


In [ ]:
import os

# Must run BEFORE any `import torch` in this kernel (CUDA_VISIBLE_DEVICES is read once).
# Changing GPU_IDS or PILOT_MODE requires restarting the kernel.
GPU_IDS = "4,5"  # physical GPU ids dedicated to this session, e.g. "0" or "4,5"
PILOT_MODE = "small_1gpu_pair"  # "small_1gpu_pair" (Qwen-3B + Gemma-1B side by side) or "large_2gpu_single"
LARGE_MODEL_KEY = "gemma-3-4b-it"  # used only when PILOT_MODE == "large_2gpu_single": "qwen2.5-7b-instruct" or "gemma-3-4b-it"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS


In [ ]:
# # For local GPU environment
# !pip install "outlines[transformers]==1.3.3" transformers bitsandbytes accelerate datasets huggingface_hub python-dotenv

import os
import sys
from pprint import pprint

import torch

from scripts.local_qa_generation import (  # noqa: E402
    MODEL_REGISTRY,
    PREGUNTAS_COMUNES,
    get_available_devices,
    load_local_model,
    process_full_dataset_local,
    process_single_context_local,
)

/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
from huggingface_hub import login

# Reads HUGGINGFACE_TOKEN from a .env file at the repo root (never commit the token).
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise RuntimeError(
        "HUGGINGFACE_TOKEN not found. Add it to a .env file at the repo root."
    )
login(hf_token)


## 0. Sanity check: is `outlines` installed and working?

Quick standalone check, independent of the GPU models loaded later, so a broken `outlines` install fails fast here instead of deep into section 3/4.

In [3]:
import outlines
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# print("outlines version:", outlines.__version__)


class _SanityCheckSchema(BaseModel):
    ok: bool


# CPU-only smoke test of schema-constrained generation, no GPU model required.
_sanity_model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2"),
    AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2"),
)
_sanity_result = _sanity_model("Reply with JSON.", _SanityCheckSchema)
print("outlines constrained-generation smoke test output:", _sanity_result)

Loading weights: 100%|██████████| 29/29 [00:00<00:00, 21959.71it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


outlines constrained-generation smoke test output: {"ok": false }


## 1. Load raw dataset and apply the same 82–150 word filter

In [4]:
from datasets import load_dataset

MIN_WORDS = 82
MAX_WORDS = 150

dataset_path = "LeninGF/autotrain-data-robberyclassification"
ds = load_dataset(dataset_path)


def count_words(sample):
    sample["word_count"] = len(sample["relato"].split())
    return sample


ds = ds.map(count_words, batched=False)
filtered_ds = ds["train"].filter(
    lambda batch: [MIN_WORDS <= x <= MAX_WORDS for x in batch["word_count"]],
    batched=True,
    batch_size=1000,
)
filtered_ds

Dataset({
    features: ['relato', 'labels', 'word_count'],
    num_rows: 174594
})

## 2. Select a pilot subset


In [5]:
PILOT_SIZE = 50
pilot_indices = list(range(min(PILOT_SIZE, len(filtered_ds))))
pilot_ds = filtered_ds.select(pilot_indices)


def pilot_context_id(i):
    """Map pilot_ds row i back to its original filtered_ds index."""
    return f"context_{pilot_indices[i]}"


pilot_ds


Dataset({
    features: ['relato', 'labels', 'word_count'],
    num_rows: 50
})

## 3. Load candidate models (per `PILOT_MODE`)

- `small_1gpu_pair`: `qwen2.5-3b-instruct` on logical `cuda:0` + `gemma-3-1b-it` on logical `cuda:1`, loaded side by side
- `large_2gpu_single`: `LARGE_MODEL_KEY` (`qwen2.5-7b-instruct` or `gemma-3-4b-it`) alone, balanced across both visible GPUs


In [ ]:
devices = get_available_devices()
print(f"Available devices (logical, after CUDA_VISIBLE_DEVICES remap): {devices}")

qwen_model = None
gemma_model = None
large_model = None

if PILOT_MODE == "small_1gpu_pair":
    if len(devices) < 2:
        raise RuntimeError("small_1gpu_pair needs 2 visible GPUs; set GPU_IDS to two ids and restart the kernel.")
    qwen_model = load_local_model("qwen2.5-3b-instruct", gpu_ids=[0], quantize_4bit=True)
    gemma_model = load_local_model("gemma-3-1b-it", gpu_ids=[1], quantize_4bit=True)
elif PILOT_MODE == "large_2gpu_single":
    entry = MODEL_REGISTRY[LARGE_MODEL_KEY]
    if len(devices) != entry["num_gpus"]:
        raise RuntimeError(
            f"{LARGE_MODEL_KEY} needs {entry['num_gpus']} visible GPUs; "
            "set GPU_IDS accordingly and restart the kernel."
        )
    large_model = load_local_model(LARGE_MODEL_KEY, gpu_ids=list(range(len(devices))), quantize_4bit=True)
else:
    raise ValueError(f"Unknown PILOT_MODE '{PILOT_MODE}'")

Available devices: ['cuda:0']
Loading Qwen/Qwen2.5-3B-Instruct on cuda:0 ...


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 421.64it/s]


## 4. Run the pilot generation

- `small_1gpu_pair`: each model writes to its own JSONL file for side-by-side comparison in section 5
- `large_2gpu_single`: runs a 1-context sanity check first for `gemma-3-4b-it` (untested outlines + `AutoProcessor` combination), then the full pilot subset


In [ ]:
PILOT_QWEN_FILE = os.path.join("dataset", "pilot_qwen2.5-3b.json")
PILOT_GEMMA_FILE = os.path.join("dataset", "pilot_gemma-3-1b.json")
PILOT_LARGE_FILE = os.path.join("dataset", f"pilot_{LARGE_MODEL_KEY}.json")

if PILOT_MODE == "small_1gpu_pair":
    process_full_dataset_local(
        dataset=pilot_ds,
        output_file=PILOT_QWEN_FILE,
        model=qwen_model,
        questions=PREGUNTAS_COMUNES,
        checkpoint_interval=10,
        context_id_fn=pilot_context_id,
    )
    process_full_dataset_local(
        dataset=pilot_ds,
        output_file=PILOT_GEMMA_FILE,
        model=gemma_model,
        questions=PREGUNTAS_COMUNES,
        checkpoint_interval=10,
        context_id_fn=pilot_context_id,
    )
elif PILOT_MODE == "large_2gpu_single":
    if LARGE_MODEL_KEY == "gemma-3-4b-it":
        # Fails fast if outlines does not support Gemma3ForConditionalGeneration + AutoProcessor.
        sanity_result = process_single_context_local(
            pilot_ds[0]["relato"], PREGUNTAS_COMUNES[:1], large_model, context_id="sanity_check"
        )
        print("Sanity check passed:", sanity_result)

    process_full_dataset_local(
        dataset=pilot_ds,
        output_file=PILOT_LARGE_FILE,
        model=large_model,
        questions=PREGUNTAS_COMUNES,
        checkpoint_interval=10,
        context_id_fn=pilot_context_id,
    )

/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=320) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Error (CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasCreate(handle)`), reintentando (1/2)...
Error crítico en contexto 0: Fallo después de 2 reintentos


/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=406) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Error (CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasCreate(handle)`), reintentando (1/2)...
Error crítico en contexto 1: Fallo después de 2 reintentos


/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=331) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


...

/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=342) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Error (CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasCreate(handle)`), reintentando (1/2)...
Error crítico en contexto 3: Fallo después de 2 reintentos


/opt/conda/envs/pyt-eqa-fge/lib/python3.11/site-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=405) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Error (CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasCreate(handle)`), reintentando (1/2)...


KeyboardInterrupt: 

## 5. Compare pilot output quality

Only applicable when `PILOT_MODE == "small_1gpu_pair"`. Same analysis patterns as `dataset_xplore_and_upload.ipynb`: `impossible_find_answer`/`is_impossible` value counts and answer-length distribution, side by side for the two models.

In [ ]:
if PILOT_MODE != "small_1gpu_pair":
    raise RuntimeError("Section 5 compares Qwen vs Gemma pilot outputs; only meaningful for PILOT_MODE == 'small_1gpu_pair'.")

from datasets import load_dataset as load_jsonl_dataset

df_qwen = load_jsonl_dataset("json", data_files=PILOT_QWEN_FILE, split="train").to_pandas()
df_gemma = load_jsonl_dataset("json", data_files=PILOT_GEMMA_FILE, split="train").to_pandas()

for name, df in [("Qwen2.5-3B-Instruct", df_qwen), ("Gemma-3-1B-IT", df_gemma)]:
    df["answer_text_number_words"] = df["answer_text"].apply(lambda x: len(str(x).split()))
    print(f"\n=== {name} ===")
    print(df["impossible_find_answer"].value_counts())
    print(df["answer_text_number_words"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, name, df in zip(axes, ["Qwen2.5-3B-Instruct", "Gemma-3-1B-IT"], [df_qwen, df_gemma]):
    sns.histplot(df["answer_text_number_words"], bins=30, kde=True, ax=ax)
    ax.set_title(f"Answer word count — {name}")
    ax.set_xlabel("Number of words")
plt.tight_layout()
plt.show()

In [ ]:
# Spot-check a few samples from each model manually before deciding
pprint(df_qwen[["question", "answer_text", "impossible_find_answer"]].sample(5))
pprint(df_gemma[["question", "answer_text", "impossible_find_answer"]].sample(5))

## 6. Decision

Fill in after reviewing the pilot results on the GPU machine:

- Winning model for the full run: **TBD**
- Reasoning: **TBD**

Before committing to a full run, compare throughput across model configs with:

```bash
python scripts/benchmark_local_gpu.py --model <qwen2.5-3b-instruct|qwen2.5-7b-instruct|gemma-3-1b-it|gemma-3-4b-it> \
    --gpu-ids <e.g. 0 or 4,5> --num-samples 5
```

Once decided, run the full-scale build with:

```bash
python scripts/build_dataset_local_gpu.py --model <qwen2.5-3b-instruct|qwen2.5-7b-instruct|gemma-3-1b-it|gemma-3-4b-it> \
    --gpu-ids <e.g. 0 or 4,5> --output-file dataset/dataset_squad_v2_localgpu.json
```
